# 01b — EDA PADBen đã chuẩn hóa
Bước 0 dùng Humarin + Task 1/2/3. Task 4/5 chỉ đánh giá trên nhóm nguồn thuộc test.
**Task 3: cả hai loại đều AI-touched=1. Task 4: cả hai mức paraphrase đều 1.
Task 5: human=0, deep paraphrase=1.** Giữ `original_task_label` để phân tích tác vụ gốc.
Notebook kiểm tra file đã xử lý, không ánh xạ lại nhãn từ raw.

## Môi trường và đường dẫn
Chọn kernel Python đã cài dependency của repo. Notebook không tự cài/upgrade PyTorch.
Chạy các ô từ trên xuống. Có thể cấu hình `CATEX_PROJECT_ROOT`, `CATEX_DATA_DIR`, `CATEX_OUTPUT_ROOT` hoặc sửa ô dưới.
Không đưa `.env` lên notebook. Các notebook này không gọi Web Search hay LLM.


In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import display

# Local: mở notebook từ repo hoặc notebooks/. Kaggle: sửa PROJECT_ROOT trỏ tới code đã giải nén.
PROJECT_ROOT = os.environ.get("CATEX_PROJECT_ROOT", "")
if PROJECT_ROOT:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
else:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "src/models/runtime.py").is_file()), None)
if ROOT is None or not (ROOT / "src/models/runtime.py").is_file():
    raise FileNotFoundError("Đặt PROJECT_ROOT thành thư mục chứa src/ và notebooks/.")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

# Kaggle: sửa thành đường dẫn thư mục split hiển thị trong Input; không ghi vào Input.
DATA_DIR = Path(os.environ.get("CATEX_DATA_DIR", str(ROOT / "data/splits"))).resolve()
DEFAULT_OUTPUT = Path("/kaggle/working/catex") if Path("/kaggle").exists() else ROOT
OUTPUT_ROOT = Path(os.environ.get("CATEX_OUTPUT_ROOT", str(DEFAULT_OUTPUT))).resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
from notebooks.workflow import inspect_csv, run_script, read_json, require_checkpoint
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print("Code:", ROOT, "\nData:", DATA_DIR, "\nOutput:", OUTPUT_ROOT)

from notebooks.eda_tools import profile_csv, text_features, normalize_text
REPORT_DIR=OUTPUT_ROOT / "evaluation/eda_results/notebook_01b"
REPORT_DIR.mkdir(parents=True,exist_ok=True)


In [ ]:
SEED=42
summaries=[]
for split in ("train","val","test"):
    summary,_=profile_csv(DATA_DIR / f"tang0_{split}.csv",["sentence"],sample_size=2000,seed=SEED)
    summaries.append(summary)
display(pd.DataFrame(summaries))


## Task 4/5 — phân bố nhãn và nguồn

In [ ]:
tasks={}
for task in (4,5):
    frame=pd.read_csv(DATA_DIR / f"padben_task{task}_test.csv",dtype={"group_id":str})
    assert not frame.empty and frame.sentence.notna().all()
    assert frame["split"].eq("test").all()
    assert set(frame.label)==({1} if task==4 else {0,1})
    assert frame.task.eq(task).all()
    tasks[task]=frame
    print(f"Task {task}: {len(frame):,} dòng, {frame.group_id.nunique():,} nhóm")
    display(pd.crosstab(frame.original_task_label,frame.label))
    display(frame.sample(min(4,len(frame)),random_state=SEED)[["sentence","label","original_task_label","sentence_position"]])


## Kiểm tra nhóm nguồn với train/validation

In [ ]:
heldout=set().union(*(set(frame.group_id) for frame in tasks.values()))
for split in ("train","val"):
    overlap=set()
    for chunk in pd.read_csv(DATA_DIR / f"tang0_{split}.csv",usecols=["group_id"],dtype=str,chunksize=100000):
        overlap.update(heldout & set(chunk.group_id))
    print(split,"overlapping groups:",len(overlap))
    assert not overlap, "Task 4/5 chứa nhóm đã dùng để train/chọn model."
print("Đây là kiểm tra group với bước 0; audit toàn bộ text vẫn là bước riêng.")


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,(task,frame) in zip(axes,tasks.items()):
    for original_label,part in frame.groupby("original_task_label"):
        ax.hist(part.sentence.str.split().str.len(),bins=25,alpha=.5,label=f"original={original_label}")
    ax.set_title(f"PADBen Task {task}");ax.set_xlabel("Số từ");ax.legend()
plt.tight_layout();plt.show()


## 4. Đếm chính xác nguồn × nhãn trong dữ liệu bước 0
Các thống kê này quét toàn file. Task 1/2/3 không còn là tập test độc lập vì đã góp vào train.
Độ lệch nhãn theo nguồn cần được đưa vào phần hạn chế khi giải thích model AI-touched.

In [ ]:
source_rows=[]
for split,summary in zip(("train","val","test"),summaries):
    source_rows.extend({"split":split,**row} for row in summary["source_labels"])
source_table=pd.DataFrame(source_rows)
display(source_table.pivot_table(index=["split","source"],columns="label",values="rows",fill_value=0))
source_table.to_csv(REPORT_DIR / "step0_source_labels.csv",index=False)
fig,ax=plt.subplots(figsize=(9,4))
source_table[source_table.split.eq("train")].pivot(index="source",columns="label",values="rows").fillna(0).plot.bar(stacked=True,ax=ax)
ax.set_title("Train bước 0: nguồn × nhãn");fig.tight_layout();fig.savefig(REPORT_DIR / "train_source_labels.png");plt.show()


## 5. Tính toàn vẹn Task 4/5 — toàn bộ bản test đã xuất
Kiểm tra cặp có đủ hai vị trí, nhãn chuẩn hóa khớp nhãn gốc và text trùng/nhãn mâu thuẫn.
Không giả định mỗi group chỉ có một cặp. Task 4: nhãn gốc phân biệt độ sâu, không phải human/AI.

In [ ]:
integrity=[]
for task,frame in tasks.items():
    text=frame.sentence.map(normalize_text)
    sizes=frame.groupby("pair_id").size()
    positions=frame.groupby("pair_id").sentence_position.apply(lambda s:set(s)=={0,1} and len(s)==2)
    expected=pd.Series(1,index=frame.index) if task==4 else frame.original_task_label
    row={"task":task,"rows":len(frame),"groups":frame.group_id.nunique(),"pairs":frame.pair_id.nunique(),
        "missing_text":int(frame.sentence.isna().sum()),"blank_text":int(frame.sentence.fillna('').str.strip().eq('').sum()),
        "pairs_not_two_rows":int(sizes.ne(2).sum()),"bad_pair_positions":int((~positions).sum()),
        "label_mapping_mismatch":int(frame.label.ne(expected).sum()),"duplicate_text_rows":int(text.duplicated().sum()),
        "conflicting_text_labels":int(pd.DataFrame({"text":text,"label":frame.label}).groupby("text").label.nunique().gt(1).sum())}
    integrity.append(row)
    print("Task",task,"original label × position")
    display(pd.crosstab(frame.original_task_label,frame.sentence_position))
display(pd.DataFrame(integrity));pd.DataFrame(integrity).to_csv(REPORT_DIR / "task_integrity.csv",index=False)


## 6. Độ dài và dấu hiệu văn phong theo nhãn tác vụ gốc
Toàn bộ Task 4/5 đã xuất được dùng ở đây. So sánh các mức trong từng task;
không gọi mọi chênh lệch là do độ sâu, vì nội dung và quá trình tạo dữ liệu cũng ảnh hưởng.

In [ ]:
all_features=[]
for task,frame in tasks.items():
    f=text_features(frame.sentence);f["task"]=task;f["original_task_label"]=frame.original_task_label.to_numpy();f["label"]=frame.label.to_numpy()
    all_features.append(f)
all_features=pd.concat(all_features,ignore_index=True)
display(all_features.groupby(["task","original_task_label"])[["words","characters","unique_word_ratio","punctuation_ratio"]].describe(percentiles=[.5,.9,.95,.99]))
all_features.to_csv(REPORT_DIR / "task_text_features.csv",index=False)
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,(task,frame) in zip(axes,tasks.items()):
    for label,part in frame.groupby("original_task_label"):
        values=np.sort(part.sentence.str.split().str.len())
        ax.plot(values,np.arange(1,len(values)+1)/len(values),label=f"original={label}")
    ax.set(title=f"Task {task}: ECDF độ dài",xlabel="Số từ",ylabel="ECDF");ax.legend()
fig.tight_layout();fig.savefig(REPORT_DIR / "task_length_ecdf.png");plt.show()


## 7. So sánh ghép cặp theo pair_id — không nhầm vị trí với nhãn
Ghép bằng `original_task_label`, không coi vế trái luôn là human. Task 4 so mức sâu (1) với mức nông (0);
Task 5 so deep (1) với human (0). Chênh lệch số từ không chứng minh mức độ bảo toàn ý nghĩa.

In [ ]:
paired_reviews=[]
for task,frame in tasks.items():
    valid_ids=frame.groupby("pair_id").original_task_label.apply(lambda s:len(s)==2 and set(s)=={0,1})
    valid=frame[frame.pair_id.isin(valid_ids[valid_ids].index)]
    pairs=valid.pivot(index="pair_id",columns="original_task_label",values="sentence").rename(columns={0:"original_label_0",1:"original_label_1"})
    pairs["word_delta_1_minus_0"]=pairs.original_label_1.str.split().str.len()-pairs.original_label_0.str.split().str.len()
    pairs["exact_equal"]=pairs.original_label_0.map(normalize_text).eq(pairs.original_label_1.map(normalize_text))
    print("Task",task,"pairs excluded:",int((~valid_ids).sum()))
    display(pairs[["word_delta_1_minus_0","exact_equal"]].describe())
    review=pairs.sample(min(8,len(pairs)),random_state=SEED).reset_index();review["task"]=task
    paired_reviews.append(review);display(review)
    pairs.to_csv(REPORT_DIR / f"task{task}_paired_lengths.csv")
pd.concat(paired_reviews,ignore_index=True).to_csv(REPORT_DIR / "paired_manual_review.csv",index=False)


## 8. Overlap giữa Task 4 và Task 5
Dùng chung group hoặc câu trong **hai tập test** không phải rò rỉ train/test, nhưng làm các phép đo không độc lập.
Không cộng số dòng hai task rồi coi tất cả là mẫu độc lập để đưa ra khoảng tin cậy.

In [ ]:
group_overlap=set(tasks[4].group_id)&set(tasks[5].group_id)
text_overlap=set(tasks[4].sentence.map(normalize_text))&set(tasks[5].sentence.map(normalize_text))
overlap={"shared_groups_task4_task5":len(group_overlap),"shared_texts_task4_task5":len(text_overlap)}
display(overlap)
(REPORT_DIR / "task_overlap.json").write_text(json.dumps(overlap,indent=2),encoding="utf-8")


## Cách đánh giá đúng
Task 4 chỉ có lớp 1: báo recall/detection rate theo `original_task_label`, không dùng AUC.
Task 5 có hai lớp: báo precision, recall, F1, AUC và confusion matrix.
Không chỉnh ngưỡng hoặc fine-tune bằng Task 4/5 test. Do Task 1/2/3 đã có trong train,
đây là kiểm tra **tác vụ/độ sâu và nhóm nguồn chưa thấy trong PADBen**, không phải hoàn toàn chưa thấy dataset PADBen.
Chạy đánh giá bằng notebook 05 sau khi có checkpoint bước 0.